# Writer Summaries Benchmark

Use this notebook for the paper's freelance writer summaries.

This notebook treats `benchmark_llm_summarization-main/writer_summaries.json` as the source of truth. It generates AI summaries for the exact same released articles, scores AI summaries against freelance writer references, and exports a pairwise review template.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import time

import pandas as pd
import requests
from bert_score import score as bert_score
from nltk.translate.meteor_score import single_meteor_score
from rouge_score import rouge_scorer


## Configuration


In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "spec.md").exists() and (PROJECT_ROOT.parent / "spec.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

def load_env_file(env_file):
    if not env_file.exists():
        return

    for line in env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

load_env_file(PROJECT_ROOT / ".env")

# 109 are all the unique writer summary articles.
NUM_UNIQUE_ARTICLES = 109

# True = use only articles that are in the paper's released pairwise evaluation file.
USE_PAIRWISE_ARTICLE_SUBSET = True

# Human-eval template size control
WRITER_REFERENCES_PER_ARTICLE_FOR_REVIEW = 1

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_TAGS_URL = "http://localhost:11434/api/tags"
OPENAI_URL = "https://api.openai.com/v1/responses"
TEMPERATURE = 0.3

MODEL_CONFIGS = [
    {"provider": "ollama", "model": "llama3.1:latest"},
    {"provider": "ollama", "model": "gemma2:9b"},
    {"provider": "openai", "model": os.environ.get("OPENAI_MODEL", "gpt-4o-mini")},
]

PAPER_DATA_DIR = PROJECT_ROOT / "benchmark_llm_summarization-main"
WRITER_SUMMARIES_FILE = PAPER_DATA_DIR / "writer_summaries.json"
PAIRWISE_RESULTS_FILE = PAPER_DATA_DIR / "pairwise_evaluation_results.json"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
RESULTS_DIR = PROJECT_ROOT / "results"
OUTPUT_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / f"{RUN_ID}_writer_summaries_generations.jsonl"
RESULTS_FILE = RESULTS_DIR / f"{RUN_ID}_writer_metric_scores.csv"
BLIND_REVIEW_FILE = RESULTS_DIR / f"{RUN_ID}_writer_blind_human_eval.json"
LABELED_REVIEW_FILE = RESULTS_DIR / f"{RUN_ID}_writer_labeled_human_eval_key.json"

print(f"Run ID: {RUN_ID}")
print(f"Unique article limit: {NUM_UNIQUE_ARTICLES}")
for model_config in MODEL_CONFIGS:
    print(f"- {model_config['provider']} / {model_config['model']}")


## Verify Model Setup


In [ ]:
ollama_response = requests.get(OLLAMA_TAGS_URL, timeout=10)
ollama_response.raise_for_status()

installed_ollama_models = [model["name"] for model in ollama_response.json().get("models", [])]
required_ollama_models = [m["model"] for m in MODEL_CONFIGS if m["provider"] == "ollama"]
missing_ollama_models = [name for name in required_ollama_models if name not in installed_ollama_models]
if missing_ollama_models:
    raise ValueError(f"Missing Ollama models: {missing_ollama_models}")

if any(m["provider"] == "openai" for m in MODEL_CONFIGS) and not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("Missing OPENAI_API_KEY in .env")

print("Installed Ollama models:", installed_ollama_models)
print("OpenAI key loaded:", bool(os.environ.get("OPENAI_API_KEY")))


## Load Writer Summary Articles

`writer_summaries.json` has multiple writer summaries for some `article_id` values. This notebook generates AI summaries once per unique article, then joins those generated summaries back to every writer summary for scoring and pairwise review.


In [ ]:
def load_writer_data():
    if not WRITER_SUMMARIES_FILE.exists():
        raise FileNotFoundError(f"Missing writer summaries file: {WRITER_SUMMARIES_FILE}")

    writer_rows = json.loads(WRITER_SUMMARIES_FILE.read_text(encoding="utf-8"))
    df_writers = pd.DataFrame(writer_rows).rename(
        columns={
            "article_id": "id",
            "summary": "writer_summary",
        }
    )
    df_writers["dataset"] = "writer_summaries"
    df_writers["reference_type"] = "freelance_writer"
    df_writers["writer_reference_id"] = df_writers.groupby("id").cumcount() + 1

    if PAIRWISE_RESULTS_FILE.exists():
        pairwise_rows = json.loads(PAIRWISE_RESULTS_FILE.read_text(encoding="utf-8"))
        pairwise_article_ids = {row["article_id"] for row in pairwise_rows}
        df_writers["in_pairwise_subset"] = df_writers["id"].isin(pairwise_article_ids)
    else:
        df_writers["in_pairwise_subset"] = False

    return df_writers


df_writer_references_all = load_writer_data()
if USE_PAIRWISE_ARTICLE_SUBSET:
    df_writer_references_all = df_writer_references_all[df_writer_references_all["in_pairwise_subset"]].copy()

selected_article_ids = df_writer_references_all["id"].drop_duplicates().head(NUM_UNIQUE_ARTICLES)
df_writer_references = df_writer_references_all[df_writer_references_all["id"].isin(selected_article_ids)].copy()

df_unique_articles_all = (
    df_writer_references_all.groupby("id", as_index=False)
    .agg(
        article=("article", "first"),
        writer_summary_count=("writer_summary", "size"),
        in_pairwise_subset=("in_pairwise_subset", "first"),
    )
)
df_unique_articles_all["article_word_count"] = df_unique_articles_all["article"].str.split().str.len()

df_articles_for_generation = df_unique_articles_all[df_unique_articles_all["id"].isin(selected_article_ids)].copy()

print(f"Available unique writer articles: {len(df_unique_articles_all)}")
print(f"Selected unique articles for generation: {len(df_articles_for_generation)}")
print(f"Selected writer reference rows for scoring: {len(df_writer_references)}")

display(df_unique_articles_all[["id", "writer_summary_count", "in_pairwise_subset", "article_word_count", "article"]].head())
df_articles_for_generation[["id", "article", "writer_summary_count", "article_word_count"]]


## Generate AI Summaries


In [ ]:
def build_prompt(article):
    return "\n".join([
        "Article:",
        article,
        "",
        "Summarize the article in around 50 words.",
        "",
        "Summary:",
    ])


def generate_ollama_summary(model_name, article):
    payload = {
        "model": model_name,
        "prompt": build_prompt(article),
        "stream": False,
        "options": {"temperature": TEMPERATURE},
    }
    response = requests.post(OLLAMA_URL, json=payload, timeout=180)
    response.raise_for_status()
    return response.json()["response"].strip()


def extract_openai_text(response_json):
    if "output_text" in response_json:
        return response_json["output_text"].strip()

    parts = []
    for output_item in response_json.get("output", []):
        for content_item in output_item.get("content", []):
            if content_item.get("type") == "output_text":
                parts.append(content_item.get("text", ""))

    text = "".join(parts).strip()
    if not text:
        raise ValueError(f"Could not find output text in OpenAI response: {response_json}")
    return text


def generate_openai_summary(model_name, article):
    payload = {
        "model": model_name,
        "input": build_prompt(article),
        "temperature": TEMPERATURE,
    }
    headers = {
        "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}",
        "Content-Type": "application/json",
    }
    response = requests.post(OPENAI_URL, headers=headers, json=payload, timeout=180)
    response.raise_for_status()
    return extract_openai_text(response.json())


def generate_summary(model_config, article):
    if model_config["provider"] == "ollama":
        return generate_ollama_summary(model_config["model"], article)
    if model_config["provider"] == "openai":
        return generate_openai_summary(model_config["model"], article)
    raise ValueError(f"Unknown provider: {model_config['provider']}")


In [ ]:
records = []
for model_config in MODEL_CONFIGS:
    for article_record in df_articles_for_generation.to_dict(orient="records"):
        start_time = time.time()
        generated_summary = generate_summary(model_config, article_record["article"])
        elapsed_seconds = round(time.time() - start_time, 2)
        records.append({
            "run_id": RUN_ID,
            "id": article_record["id"],
            "dataset": "writer_summaries",
            "provider": model_config["provider"],
            "model": model_config["model"],
            "article": article_record["article"],
            "generated_summary": generated_summary,
            "elapsed_seconds": elapsed_seconds,
        })

df_outputs = pd.DataFrame(records)
df_outputs[["provider", "model", "id", "generated_summary", "elapsed_seconds"]]


## Save Generated Summaries


In [ ]:
with OUTPUT_FILE.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved raw generations JSONL to {OUTPUT_FILE}")


## Score AI Summaries Against Writer References


In [ ]:
class NoWordNet:
    def synsets(self, word):
        return []

no_wordnet = NoWordNet()
rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

df_metric_inputs = df_outputs.merge(
    df_writer_references[["id", "writer_reference_id", "writer_summary", "reference_type"]],
    on="id",
    how="inner",
)
df_metric_inputs["reference_summary"] = df_metric_inputs["writer_summary"]

metric_rows = []
for record in df_metric_inputs.to_dict(orient="records"):
    reference = record["reference_summary"]
    generated = record["generated_summary"]
    rouge_scores = rouge.score(reference, generated)
    meteor = single_meteor_score(reference.split(), generated.split(), wordnet=no_wordnet)
    metric_rows.append({
        "run_id": record["run_id"],
        "id": record["id"],
        "writer_reference_id": record["writer_reference_id"],
        "dataset": record["dataset"],
        "provider": record["provider"],
        "model": record["model"],
        "reference_type": record["reference_type"],
        "rouge1_fmeasure": rouge_scores["rouge1"].fmeasure,
        "rouge2_fmeasure": rouge_scores["rouge2"].fmeasure,
        "rougeL_fmeasure": rouge_scores["rougeL"].fmeasure,
        "meteor": meteor,
    })

df_scores = pd.DataFrame(metric_rows)
df_scores


## Compute BERTScore


In [ ]:
bert_precision, bert_recall, bert_f1 = bert_score(
    df_metric_inputs["generated_summary"].tolist(),
    df_metric_inputs["reference_summary"].tolist(),
    lang="en",
    verbose=True,
)

df_scores["bertscore_precision"] = bert_precision.tolist()
df_scores["bertscore_recall"] = bert_recall.tolist()
df_scores["bertscore_f1"] = bert_f1.tolist()
df_scores.to_csv(RESULTS_FILE, index=False)

df_combined = df_metric_inputs.merge(
    df_scores,
    on=["run_id", "id", "writer_reference_id", "dataset", "provider", "model", "reference_type"],
    how="left",
)

summary_table = (
    df_scores.groupby(["provider", "model"])[["rouge1_fmeasure", "rouge2_fmeasure", "rougeL_fmeasure", "meteor", "bertscore_f1"]]
    .mean()
    .reset_index()
)
summary_table


## Export Human Evaluation JSON Files

This creates one blind file for evaluators and one labeled key file for decoding results later.


In [ ]:
review_inputs = df_combined.copy()
review_inputs = (
    review_inputs.sort_values(["id", "provider", "model", "writer_reference_id"])
    .groupby(["id", "provider", "model"], as_index=False, group_keys=False)
    .head(WRITER_REFERENCES_PER_ARTICLE_FOR_REVIEW)
)

blind_items = []
labeled_items = []
for row_number, record in enumerate(review_inputs.to_dict(orient="records"), start=1):
    writer_first = row_number % 2 == 0
    if writer_first:
        summary_a = record["reference_summary"]
        summary_b = record["generated_summary"]
        summary_a_source = "writer"
        summary_b_source = "ai"
    else:
        summary_a = record["generated_summary"]
        summary_b = record["reference_summary"]
        summary_a_source = "ai"
        summary_b_source = "writer"

    review_id = f"{RUN_ID}_{row_number:04d}"
    blind_items.append({
        "review_id": review_id,
        "article": record["article"],
        "summary_a": summary_a,
        "summary_b": summary_b,
        "overall_preference": None,
        "factuality_preference": None,
        "coverage_preference": None,
        "notes": None,
    })

    labeled_items.append({
        "review_id": review_id,
        "run_id": record["run_id"],
        "article_id": record["id"],
        "writer_reference_id": int(record["writer_reference_id"]),
        "provider": record["provider"],
        "model": record["model"],
        "article": record["article"],
        "summary_a": summary_a,
        "summary_b": summary_b,
        "summary_a_source": summary_a_source,
        "summary_b_source": summary_b_source,
        "writer_summary": record["reference_summary"],
        "ai_summary": record["generated_summary"],
        "overall_preference": None,
        "factuality_preference": None,
        "coverage_preference": None,
        "notes": None,
    })

BLIND_REVIEW_FILE.write_text(json.dumps(blind_items, ensure_ascii=False, indent=2), encoding="utf-8")
LABELED_REVIEW_FILE.write_text(json.dumps(labeled_items, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Blind review items: {len(blind_items)}")
print(f"Labeled review key items: {len(labeled_items)}")
blind_items[:2]


## Exported Files


In [ ]:
print(f"Metric scores CSV: {RESULTS_FILE}")
print(f"Blind human-eval JSON: {BLIND_REVIEW_FILE}")
print(f"Labeled human-eval key JSON: {LABELED_REVIEW_FILE}")
print(f"Raw model generations JSONL: {OUTPUT_FILE}")
